In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Labs"
)

xpt_file = DATA_DIR / "TCHOL_I.xpt"
csv_file = DATA_DIR / "TCHOL_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 TCHOL_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXTC",
    "LBDTCSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TCHOL_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 8021:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,021 rows x 3 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# TCHOL_I is a MIXED integer / decimal laboratory file.
#
#
# SEQN
#     Respondent identifier
#     -> whole-number
#
#
# LBXTC
#     Serum total cholesterol in mg/dL
#     -> released as whole-number values
#
#
# LBDTCSI
#     Serum total cholesterol in mmol/L
#     -> genuine continuous decimal
#
#
# IMPORTANT:
#
# Only:
#
#     SEQN
#     LBXTC
#
# should lose unnecessary ".0".
#
#
# LBDTCSI stays FLOAT exactly as released.
#

DECIMAL_COLUMNS = [
    "LBDTCSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "LBXTC"
]


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly 1 genuine decimal field."
    )


if len(INTEGER_COLUMNS) != 2:

    raise ValueError(
        "Expected exactly 2 whole-number fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
#
# Your attached TCHOL_I has NONE.
#
# Therefore DO NOT perform any tiny -> zero replacement.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 3 TCHOL_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total > 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in TCHOL_I."
    )


if true_zero_total > 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected zero values "
        "found in TCHOL_I."
    )


print(
    "PASS: TCHOL_I contains 0 tiny artifacts "
    "and 0 ordinary zeros."
)


# ============================================================
# 6. NO ZERO CORRECTION REQUIRED
# ============================================================

#
# No numerical correction is required.
#
# The original XPT values are therefore the
# exact fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero restoration required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY as a diagnostic test.
#
# No value is modified.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: LBDTCSI is the only genuine "
    "decimal variable."
)


# ============================================================
# 8. LBDTCSI DECIMAL DETAILS
# ============================================================

si = df["LBDTCSI"]


si_nonmissing = int(
    si.notna()
    .sum()
)


si_missing = int(
    si.isna()
    .sum()
)


si_values = (
    si.dropna()
    .to_numpy(
        dtype=float
    )
)


si_fractional = int(
    (
        ~np.isclose(
            si_values,
            np.round(si_values),
            rtol=0,
            atol=1e-12
        )
    )
    .sum()
)


si_whole_looking = (
    si_nonmissing
    -
    si_fractional
)


si_min = float(
    si.min()
)


si_max = float(
    si.max()
)


print(
    "\n--- LBDTCSI DECIMAL DETAILS ---"
)

print(
    "Nonmissing:",
    f"{si_nonmissing:,}"
)

print(
    "Missing:",
    f"{si_missing:,}"
)

print(
    "Minimum:",
    repr(si_min)
)

print(
    "Maximum:",
    repr(si_max)
)

print(
    "Fractional observations:",
    f"{si_fractional:,}"
)

print(
    "Whole-looking observations:",
    f"{si_whole_looking:,}"
)


if si_nonmissing != 7256:

    raise ValueError(
        "Unexpected LBDTCSI nonmissing count."
    )


if si_missing != 765:

    raise ValueError(
        "Unexpected LBDTCSI missing count."
    )


if si_min != 1.99:

    raise ValueError(
        "Unexpected LBDTCSI minimum."
    )


if si_max != 14.09:

    raise ValueError(
        "Unexpected LBDTCSI maximum."
    )


if si_fractional != 7208:

    raise ValueError(
        "Unexpected number of fractional "
        "LBDTCSI observations."
    )


if si_whole_looking != 48:

    raise ValueError(
        "Unexpected number of whole-looking "
        "LBDTCSI observations."
    )


print(
    "PASS: LBDTCSI decimal structure validated."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 10. VALIDATE LBXTC IS WHOLE-NUMBER
# ============================================================

#
# LBXTC is the released mg/dL result.
#
# All populated values in this file are whole-number.
#

lbx_values = (
    df["LBXTC"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


whole_mask = np.isclose(
    lbx_values,
    np.round(lbx_values),
    rtol=0,
    atol=1e-12
)


if not whole_mask.all():

    raise ValueError(
        "LBXTC unexpectedly contains "
        "fractional values."
    )


print(
    "PASS: All populated LBXTC values "
    "are whole-number."
)


# ============================================================
# 11. CONVERT LBXTC TO Int64
# ============================================================

#
# NO .round() during conversion.
#

df["LBXTC"] = (
    df["LBXTC"]
    .astype("Int64")
)


print(
    "PASS: LBXTC converted to nullable Int64 "
    "without rounding."
)


# ============================================================
# 12. CONFIRM LBDTCSI REMAINS FLOAT
# ============================================================

if not pd.api.types.is_float_dtype(
    df["LBDTCSI"].dtype
):

    raise ValueError(
        "LBDTCSI is no longer floating-point."
    )


print(
    "PASS: LBDTCSI remains floating-point."
)


# ============================================================
# 13. CDC CHECK - LBXTC
# ============================================================

#
# CDC:
#
#     77-545 mg/dL
#     n = 7,256
#     missing = 765
#

lbx_nonmissing = int(
    df["LBXTC"]
    .notna()
    .sum()
)


lbx_missing = int(
    df["LBXTC"]
    .isna()
    .sum()
)


lbx_range_count = int(
    df["LBXTC"]
    .between(
        77,
        545
    )
    .sum()
)


lbx_min = int(
    df["LBXTC"]
    .min()
)


lbx_max = int(
    df["LBXTC"]
    .max()
)


print(
    "\n--- LBXTC ---"
)

print(
    "Nonmissing:",
    f"{lbx_nonmissing:,}"
)

print(
    "Missing:",
    f"{lbx_missing:,}"
)

print(
    "Minimum:",
    lbx_min
)

print(
    "Maximum:",
    lbx_max
)


if lbx_nonmissing != 7256:

    raise ValueError(
        "LBXTC nonmissing count differs."
    )


if lbx_missing != 765:

    raise ValueError(
        "LBXTC missing count differs."
    )


if lbx_range_count != 7256:

    raise ValueError(
        "LBXTC contains values outside 77-545."
    )


if lbx_min != 77:

    raise ValueError(
        "LBXTC minimum differs."
    )


if lbx_max != 545:

    raise ValueError(
        "LBXTC maximum differs."
    )


print(
    "PASS: LBXTC matches CDC range/count."
)


# ============================================================
# 14. CDC CHECK - LBDTCSI
# ============================================================

#
# CDC:
#
#     1.99-14.09 mmol/L
#     n = 7,256
#     missing = 765
#

si_range_count = int(
    df["LBDTCSI"]
    .between(
        1.99,
        14.09
    )
    .sum()
)


if si_range_count != 7256:

    raise ValueError(
        "LBDTCSI contains values outside "
        "CDC range."
    )


if int(
    df["LBDTCSI"]
    .notna()
    .sum()
) != 7256:

    raise ValueError(
        "LBDTCSI nonmissing count differs."
    )


if int(
    df["LBDTCSI"]
    .isna()
    .sum()
) != 765:

    raise ValueError(
        "LBDTCSI missing count differs."
    )


print(
    "PASS: LBDTCSI matches CDC range/count."
)


# ============================================================
# 15. MISSING MASK CONSISTENCY
# ============================================================

#
# LBXTC and LBDTCSI represent the same laboratory
# result in different units.
#
# Their missing positions should be identical.
#

lbx_missing_mask = (
    df["LBXTC"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


si_missing_mask = (
    df["LBDTCSI"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


mask_mismatches = int(
    np.sum(
        lbx_missing_mask
        !=
        si_missing_mask
    )
)


print(
    "\n--- LBXTC / LBDTCSI MISSING MASK ---"
)

print(
    "Mismatches:",
    mask_mismatches
)


if mask_mismatches != 0:

    raise ValueError(
        "LBXTC and LBDTCSI missing positions differ."
    )


print(
    "PASS: Missing positions match exactly."
)


# ============================================================
# 16. CDC SI CONVERSION CHECK
# ============================================================

#
# CDC documentation:
#
#     LBDTCSI = LBXTC * 0.02586
#
# Released SI result is represented to 2 decimals.
#
#
# IMPORTANT:
#
# We use Decimal + ROUND_HALF_UP ONLY to validate the
# documented conversion.
#
# We DO NOT overwrite or regenerate LBDTCSI.
#
# The released XPT value remains the fidelity source.
#

valid_mask = (
    df["LBXTC"]
    .notna()
    &
    df["LBDTCSI"]
    .notna()
)


conversion_mismatches = 0


for mg_dl, released_si in zip(
    df.loc[
        valid_mask,
        "LBXTC"
    ],
    df.loc[
        valid_mask,
        "LBDTCSI"
    ]
):

    calculated = (
        Decimal(
            str(int(mg_dl))
        )
        *
        Decimal("0.02586")
    )


    calculated_2dp = calculated.quantize(
        Decimal("0.01"),
        rounding=ROUND_HALF_UP
    )


    released_decimal = Decimal(
        str(float(released_si))
    ).quantize(
        Decimal("0.01")
    )


    if calculated_2dp != released_decimal:

        conversion_mismatches += 1


print(
    "\n--- CDC SI CONVERSION CHECK ---"
)

print(
    "Records checked:",
    f"{int(valid_mask.sum()):,}"
)

print(
    "Conversion mismatches:",
    conversion_mismatches
)


if conversion_mismatches != 0:

    raise ValueError(
        "LBDTCSI does not match documented "
        "LBXTC x 0.02586 conversion."
    )


print(
    "PASS: All 7,256 SI results match "
    "the documented CDC conversion."
)


# ============================================================
# 17. CONFIRM NO ZERO VALUES CREATED
# ============================================================

#
# Original XPT:
#
#     tiny values = 0
#     ordinary zeros = 0
#
# Conversion should therefore create no numeric zeros.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zeros were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 18. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# No numeric repair was required.
#
# Therefore every numeric value must remain identical
# to the source XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 3 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 19. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# Result:
#
# SEQN:
#     83732.0 -> 83732
#
# LBXTC:
#     173.0 -> 173
#
# LBDTCSI:
#     4.47 -> 4.47
#     5.0  -> 5.0
#
#
# Same principle applies to your Dietary decimals:
#
# genuine continuous values remain AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 20. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 21. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: 0 numeric differences across "
    "all 3 TCHOL_I variables."
)


# ============================================================
# 22. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 23. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 24. FINAL TINY VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 25. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT check LBDTCSI for ".0".
#
# LBDTCSI is a genuine continuous lab field.
#
# Only:
#
#     SEQN
#     LBXTC
#
# should have unnecessary ".0" removed.
#

print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in integer fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "SEQN or LBXTC."
)


# ============================================================
# 26. LEGITIMATE LBDTCSI .0 VALUES
# ============================================================

#
# LBDTCSI is a continuous decimal field.
#
# The attached file has 48 whole-looking values.
#
# Examples may include:
#
#     4.0
#     5.0
#
# They must remain decimal representations.
#

si_dot_zero_count = int(
    (
        raw_csv_df["LBDTCSI"]
        .ne("")
        &
        raw_csv_df["LBDTCSI"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\n--- LEGITIMATE LBDTCSI .0 VALUES ---"
)

print(
    "Count:",
    f"{si_dot_zero_count:,}"
)


if si_dot_zero_count != 48:

    raise ValueError(
        "Unexpected LBDTCSI '.0' token count."
    )


print(
    "PASS: 48 legitimate whole-looking "
    "LBDTCSI values preserved as decimals."
)


# ============================================================
# 27. EXACT LBDTCSI DECIMAL FIDELITY
# ============================================================

#
# rtol = 0
# atol = 0
#
# No numeric change is accepted.
#

expected_si = (
    expected_df["LBDTCSI"]
    .to_numpy(
        dtype=float
    )
)


exported_si = (
    roundtrip_df["LBDTCSI"]
    .to_numpy(
        dtype=float
    )
)


si_exact = np.allclose(
    expected_si,
    exported_si,
    rtol=0,
    atol=0,
    equal_nan=True
)


if not si_exact:

    raise ValueError(
        "LBDTCSI decimal fidelity failed."
    )


print(
    "\nPASS: Every LBDTCSI value "
    "preserved exactly."
)


# ============================================================
# 28. FINAL RANGE CHECK
# ============================================================

final_lbx = pd.to_numeric(
    raw_csv_df["LBXTC"]
    .where(
        raw_csv_df["LBXTC"] != "",
        np.nan
    ),
    errors="coerce"
)


final_si = pd.to_numeric(
    raw_csv_df["LBDTCSI"]
    .where(
        raw_csv_df["LBDTCSI"] != "",
        np.nan
    ),
    errors="coerce"
)


if final_lbx.min() != 77:

    raise ValueError(
        "Final LBXTC minimum changed."
    )


if final_lbx.max() != 545:

    raise ValueError(
        "Final LBXTC maximum changed."
    )


if final_si.min() != 1.99:

    raise ValueError(
        "Final LBDTCSI minimum changed."
    )


if final_si.max() != 14.09:

    raise ValueError(
        "Final LBDTCSI maximum changed."
    )


print(
    "PASS: Final cholesterol ranges preserved."
)


# ============================================================
# 29. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found in CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TCHOL_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Laboratory - Total Cholesterol"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Whole-number/ID fields:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "LBXTC nonmissing:",
    f"{lbx_nonmissing:,}"
)


print(
    "LBXTC missing:",
    f"{lbx_missing:,}"
)


print(
    "LBXTC range:",
    f"{lbx_min}-{lbx_max} mg/dL"
)


print(
    "LBDTCSI nonmissing:",
    f"{si_nonmissing:,}"
)


print(
    "LBDTCSI missing:",
    f"{si_missing:,}"
)


print(
    "LBDTCSI range:",
    "1.99-14.09 mmol/L"
)


print(
    "LBDTCSI fractional observations:",
    f"{si_fractional:,}"
)


print(
    "LBDTCSI whole-looking observations:",
    f"{si_whole_looking:,}"
)


print(
    "CDC SI conversion mismatches:",
    conversion_mismatches
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "LBDTCSI decimal differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0' in SEQN/LBXTC:",
    "0 expected"
)


print(
    "LBDTCSI '.0' values:",
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "LBDTCSI remains a genuine continuous "
    "laboratory decimal variable."
)


print(
    "No laboratory decimal value was rounded "
    "or rewritten."
)

NHANES 2015-2016 TCHOL_I XPT -> CSV
Rows: 8,021
Columns: 3
PASS: Structure = 8,021 rows x 3 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['LBDTCSI']
Whole-number fields: ['SEQN', 'LBXTC']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: TCHOL_I contains 0 tiny artifacts and 0 ordinary zeros.

PASS: No tiny-to-zero restoration required.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
LBXTC: 0 fractional observations
LBDTCSI: 7,208 fractional observations

PASS: LBDTCSI is the only genuine decimal variable.

--- LBDTCSI DECIMAL DETAILS ---
Nonmissing: 7,256
Missing: 765
Minimum: 1.99
Maximum: 14.09
Fractional observations: 7,208
Whole-looking observations: 48
PASS: LBDTCSI decimal structure validated.

--- SEQN ---
Minimum: 83732
Maximum: 93702
Missing: 0
Duplicates: 0
PASS: SEQN 83732-93702 validated.
PASS: All populated LBXTC values are whole-number.
PASS: LBXTC converted to nullable Int64 without rounding.
PASS: L